
# PPMI RNA-seq Differential Expression Stability Analysis with PyDESeq2

이 노트북은 PPMI RNA-seq `featureCounts` 파일들과 `metaDataIR3.csv`를 이용해
반복적 차등발현분석의 **편차(stability / variability)** 를 평가하기 위한 템플릿입니다.

지원 기능:
- 진단군 선택 (기본값: `PD` vs `Control`)
- 방문시점 선택 (기본값: `BL`)
- QC 통과 여부 필터 (`QCflagIR3 == "pass"`)
- 분석 방식 선택
  - `bootstrap`: 진단군별 복원추출 기반 반복 DE
  - `kfold`: Stratified K-fold 분할 기반 반복 DE
- 반복 횟수 / fold 수 지정
- gene filtering (`MIN_COUNT_GENE`)
- 유의성 판정 지표 선택 (`pvalue` 또는 `padj`)
- 유의수준 지정 (`ALPHA`)
- 반복 결과의 분포 / 유전자 안정성 요약 / Jaccard 유사도 계산

> **중요**  
> 이 노트북은 **샘플별 개별 featureCounts 파일이 들어 있는 디렉터리**를 대상으로 동작합니다.  
> 실제 실행 전 `COUNTS_DIR` 경로를 본인 환경에 맞게 수정해 주세요.


In [ ]:

# 필요 시 최초 1회 설치
# %pip install -U pydeseq2 scikit-learn pandas numpy matplotlib


In [ ]:

from __future__ import annotations

import re
import math
import json
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Iterable, Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold

from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
from pydeseq2.default_inference import DefaultInference

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)


In [ ]:

@dataclass
class AnalysisConfig:
    COUNTS_DIR: str = "./Dataset/RNA/IR3/counts/"
    METADATA_PATH: str = "./Dataset/RNA/IR3/metaDataIR3.csv"
    RESULTS_DIR: str = "./de_stability_results"

    DIAGNOSES: tuple = ("PD", "Control")
    VISITS: tuple = ("BL",)
    QC_PASS_ONLY: bool = True

    ANALYSIS_MODE: str = "kfold"   # 'bootstrap' or 'kfold'
    N_ITERATIONS_OR_FOLDS: int = 5
    RANDOM_STATE: int = 42

    CONDITION_COL: str = "condition"
    CASE_LABEL: str = "PD"
    CONTROL_LABEL: str = "Control"
    DESIGN: str = "condition"

    MIN_COUNT_GENE: int = 10

    SIGNIFICANCE_METRIC: str = "padj"
    ALPHA: float = 0.05

    N_CPUS: Optional[int] = None
    REFIT_COOKS: bool = True
    COOKS_FILTER: bool = True
    INDEPENDENT_FILTER: bool = True
    MIN_REPLICATES: int = 3

    KFOLD_SUBSET: str = "train"

    COUNT_FILE_GLOB: str = "*featureCounts*"
    QUIET: bool = True


cfg = AnalysisConfig()
print(json.dumps(asdict(cfg), indent=2, ensure_ascii=False))



## 1. 메타데이터 구조 및 기본값 확인

`metaDataIR3.csv`에서 실제 진단군 / 방문시점 / QC 값을 먼저 확인합니다.


In [ ]:

meta_raw = pd.read_csv(cfg.METADATA_PATH)

print("shape:", meta_raw.shape)
print("\nDIAGNOSIS unique:")
print(sorted(meta_raw["DIAGNOSIS"].dropna().astype(str).unique().tolist()))

print("\nCLINICAL_EVENT unique:")
print(sorted(meta_raw["CLINICAL_EVENT"].dropna().astype(str).unique().tolist()))

print("\nQCflagIR3 unique:")
print(sorted(meta_raw["QCflagIR3"].dropna().astype(str).unique().tolist()))

display(meta_raw[["PATNO", "DIAGNOSIS", "CLINICAL_EVENT", "QCflagIR3", "HudAlphaID", "Sample"]].head())



## 2. 유틸리티 함수

- featureCounts 파일명 파싱
- 개별 count 파일 읽기
- count matrix 구성
- 메타데이터 정리 및 매칭


In [ ]:

FEATURECOUNTS_SUFFIX_PATTERNS = [
    r"\.featureCounts\.GencodeV29\.txt$",
    r"\.featureCounts\.GencodeV29$",
    r"\.featureCounts-GencodeV29\.txt$",
    r"\.featureCounts-GencodeV29$",
    r"\.featureCounts.*$",
]

def strip_featurecounts_suffix(name: str) -> str:
    out = name
    for pat in FEATURECOUNTS_SUFFIX_PATTERNS:
        out = re.sub(pat, "", out)
    return out

def normalize_sample_key(x: object) -> str:
    if pd.isna(x):
        return ""
    s = str(x).strip()
    s = Path(s).name
    s = re.sub(r"\.txt$", "", s)
    s = strip_featurecounts_suffix(s)
    return s

def parse_count_filename(path: str | Path) -> Dict[str, str]:
    # 예시:
    # PPMI-Phase1-IR3.3229.BL.PP0015-6671.5104-SL-1298.featureCounts.GencodeV29
    name = Path(path).name
    sample_key = normalize_sample_key(name)
    parts = sample_key.split(".")
    parsed = {
        "file_name": name,
        "sample_key": sample_key,
        "study_release": parts[0] if len(parts) > 0 else "",
        "PATNO_file": parts[1] if len(parts) > 1 else "",
        "CLINICAL_EVENT_file": parts[2] if len(parts) > 2 else "",
        "SampleID_file": parts[3] if len(parts) > 3 else "",
        "HudAlphaID_file": parts[4] if len(parts) > 4 else "",
    }
    return parsed

def read_featurecounts_file(path: str | Path) -> pd.Series:
    path = Path(path)
    df = pd.read_csv(path, sep="\t", comment="#", low_memory=False)

    if df.empty:
        raise ValueError(f"Empty featureCounts file: {path}")

    possible_gene_cols = ["Geneid", "gene_id", "GeneID", "gene"]
    gene_col = next((c for c in possible_gene_cols if c in df.columns), df.columns[0])

    count_col = df.columns[-1]

    counts = pd.Series(df[count_col].values, index=df[gene_col].astype(str).values, name=normalize_sample_key(path.name))
    counts = pd.to_numeric(counts, errors="coerce").fillna(0).astype(int)
    return counts

def build_counts_matrix(counts_dir: str | Path, file_glob: str = "*featureCounts*") -> Tuple[pd.DataFrame, pd.DataFrame]:
    counts_dir = Path(counts_dir)
    files = sorted(counts_dir.glob(file_glob))
    if not files:
        raise FileNotFoundError(f"No files matched {file_glob!r} under {counts_dir}")

    sample_series = []
    file_records = []

    for fp in files:
        parsed = parse_count_filename(fp)
        ser = read_featurecounts_file(fp)
        sample_series.append(ser)
        file_records.append(parsed)

    counts_df = pd.DataFrame(sample_series)
    counts_df.index = [s.name for s in sample_series]
    counts_df.index.name = "sample_key"
    counts_df.columns.name = "gene"

    file_info = pd.DataFrame(file_records)
    return counts_df, file_info

def prepare_metadata(meta_df: pd.DataFrame,
                     diagnoses: Iterable[str],
                     visits: Iterable[str],
                     qc_pass_only: bool = True) -> pd.DataFrame:
    meta = meta_df.copy()

    meta["PATNO"] = meta["PATNO"].astype(str)
    meta["DIAGNOSIS"] = meta["DIAGNOSIS"].astype(str)
    meta["CLINICAL_EVENT"] = meta["CLINICAL_EVENT"].astype(str)
    meta["QCflagIR3"] = meta["QCflagIR3"].astype(str)
    meta["HudAlphaID"] = meta["HudAlphaID"].astype(str)
    meta["normalized_sample_key"] = meta["Sample"].map(normalize_sample_key)

    meta = meta[meta["DIAGNOSIS"].isin(list(diagnoses))]
    meta = meta[meta["CLINICAL_EVENT"].isin(list(visits))]
    if qc_pass_only:
        meta = meta[meta["QCflagIR3"].str.lower().eq("pass")]

    return meta.reset_index(drop=True)

def match_counts_to_metadata(counts_df: pd.DataFrame,
                             file_info: pd.DataFrame,
                             meta_df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    meta = meta_df.copy()
    files = file_info.copy()

    files["sample_key"] = files["sample_key"].astype(str)
    files["HudAlphaID_file"] = files["HudAlphaID_file"].astype(str)
    meta["normalized_sample_key"] = meta["normalized_sample_key"].astype(str)
    meta["HudAlphaID"] = meta["HudAlphaID"].astype(str)

    files = files.set_index("sample_key").loc[counts_df.index].reset_index()

    exact = files.merge(
        meta,
        left_on="sample_key",
        right_on="normalized_sample_key",
        how="left",
        suffixes=("", "_meta"),
    )

    need_fallback = exact["PATNO"].isna()
    if need_fallback.any():
        fallback = files.loc[need_fallback, ["sample_key", "HudAlphaID_file"]].merge(
            meta,
            left_on="HudAlphaID_file",
            right_on="HudAlphaID",
            how="left",
            suffixes=("", "_meta"),
        )
        fallback = fallback.set_index("sample_key")

        exact = exact.set_index("sample_key")
        fill_cols = [c for c in fallback.columns if c in exact.columns]
        for col in fill_cols:
            exact.loc[fallback.index, col] = exact.loc[fallback.index, col].combine_first(fallback[col])
        exact = exact.reset_index()

    matched = exact.copy()
    matched = matched[~matched["PATNO"].isna()].copy()

    if matched.empty:
        raise ValueError("No count files could be matched to metadata. Please check file naming and metadata fields.")

    counts_matched = counts_df.loc[matched["sample_key"]].copy()
    metadata_matched = matched.copy()

    metadata_matched["analysis_sample_id"] = metadata_matched["sample_key"].astype(str)
    metadata_matched = metadata_matched.set_index("analysis_sample_id")

    counts_matched.index = metadata_matched.index

    metadata_matched["condition"] = metadata_matched["DIAGNOSIS"].astype(str)

    return counts_matched, metadata_matched, matched

def load_and_prepare_data(cfg: AnalysisConfig) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    counts_df, file_info = build_counts_matrix(cfg.COUNTS_DIR, cfg.COUNT_FILE_GLOB)
    meta_raw = pd.read_csv(cfg.METADATA_PATH)
    meta_filtered = prepare_metadata(
        meta_raw,
        diagnoses=cfg.DIAGNOSES,
        visits=cfg.VISITS,
        qc_pass_only=cfg.QC_PASS_ONLY,
    )
    counts_matched, metadata_matched, matched_table = match_counts_to_metadata(counts_df, file_info, meta_filtered)

    metadata_matched[cfg.CONDITION_COL] = pd.Categorical(
        metadata_matched[cfg.CONDITION_COL].astype(str),
        categories=[cfg.CONTROL_LABEL, cfg.CASE_LABEL],
        ordered=True,
    )

    valid = metadata_matched[cfg.CONDITION_COL].notna()
    counts_matched = counts_matched.loc[valid]
    metadata_matched = metadata_matched.loc[valid]

    return counts_matched, metadata_matched, matched_table



## 3. 데이터 로드 및 매칭 확인


In [ ]:

# 실제 실행 전 cfg.COUNTS_DIR 경로를 반드시 수정하세요.
# 예:
# cfg.COUNTS_DIR = "/home/yourname/PPMI/longRNA/counts"
# cfg.METADATA_PATH = "/home/yourname/PPMI/metaDataIR3.csv"

# counts_all, meta_all, matched_table = load_and_prepare_data(cfg)
# print("counts shape:", counts_all.shape)
# print("metadata shape:", meta_all.shape)
# display(matched_table.head())
# display(meta_all[[cfg.CONDITION_COL, "CLINICAL_EVENT", "QCflagIR3", "PATNO", "HudAlphaID"]].head())



## 4. 분할 전략 함수

- `bootstrap`: 각 진단군 내에서 원래 샘플 수만큼 **복원추출**
- `kfold`: `StratifiedKFold`를 이용해 클래스 비율을 유지한 분할  
  - `KFOLD_SUBSET="train"`이면 각 fold의 train subset으로 DE 수행
  - `KFOLD_SUBSET="test"`이면 각 fold의 held-out subset으로 DE 수행


In [ ]:

def _make_unique_index(ids: List[str], prefix: str) -> List[str]:
    return [f"{prefix}__{i:04d}__{sid}" for i, sid in enumerate(ids)]

def make_bootstrap_subsets(counts_df: pd.DataFrame,
                           meta_df: pd.DataFrame,
                           condition_col: str,
                           n_iterations: int,
                           random_state: int = 42):
    rng = np.random.default_rng(random_state)
    groups = meta_df.groupby(condition_col, observed=True)

    for i in range(1, n_iterations + 1):
        sampled_ids = []
        for label, sub in groups:
            ids = sub.index.to_numpy()
            sampled = rng.choice(ids, size=len(ids), replace=True)
            sampled_ids.extend(sampled.tolist())

        sub_counts = counts_df.loc[sampled_ids].copy()
        sub_meta = meta_df.loc[sampled_ids].copy()

        new_index = _make_unique_index(sampled_ids, prefix=f"bootstrap_{i}")
        sub_counts.index = new_index
        sub_meta.index = new_index

        yield {
            "iteration_id": f"bootstrap_{i}",
            "subset_type": "bootstrap",
            "counts": sub_counts,
            "metadata": sub_meta,
        }

def make_kfold_subsets(counts_df: pd.DataFrame,
                       meta_df: pd.DataFrame,
                       condition_col: str,
                       n_splits: int,
                       random_state: int = 42,
                       subset: str = "train"):
    y = meta_df[condition_col].astype(str).values
    sample_ids = meta_df.index.to_numpy()

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    for fold, (train_idx, test_idx) in enumerate(skf.split(sample_ids, y), start=1):
        idx = train_idx if subset == "train" else test_idx
        selected_ids = sample_ids[idx].tolist()

        sub_counts = counts_df.loc[selected_ids].copy()
        sub_meta = meta_df.loc[selected_ids].copy()

        yield {
            "iteration_id": f"kfold_{subset}_{fold}",
            "subset_type": f"kfold_{subset}",
            "counts": sub_counts,
            "metadata": sub_meta,
        }

def generate_subsets(counts_df: pd.DataFrame,
                     meta_df: pd.DataFrame,
                     cfg: AnalysisConfig):
    mode = cfg.ANALYSIS_MODE.lower()
    if mode == "bootstrap":
        yield from make_bootstrap_subsets(
            counts_df=counts_df,
            meta_df=meta_df,
            condition_col=cfg.CONDITION_COL,
            n_iterations=cfg.N_ITERATIONS_OR_FOLDS,
            random_state=cfg.RANDOM_STATE,
        )
    elif mode == "kfold":
        yield from make_kfold_subsets(
            counts_df=counts_df,
            meta_df=meta_df,
            condition_col=cfg.CONDITION_COL,
            n_splits=cfg.N_ITERATIONS_OR_FOLDS,
            random_state=cfg.RANDOM_STATE,
            subset=cfg.KFOLD_SUBSET,
        )
    else:
        raise ValueError("ANALYSIS_MODE must be either 'bootstrap' or 'kfold'")



## 5. 단일 subset에 대한 PyDESeq2 실행 함수


In [ ]:

def filter_genes_by_min_count(counts_df: pd.DataFrame, min_count_gene: int) -> pd.DataFrame:
    keep = counts_df.sum(axis=0) >= int(min_count_gene)
    return counts_df.loc[:, keep]

def validate_subset_for_de(meta_df: pd.DataFrame,
                           condition_col: str,
                           case_label: str,
                           control_label: str) -> None:
    vc = meta_df[condition_col].astype(str).value_counts()
    if case_label not in vc.index or control_label not in vc.index:
        raise ValueError(f"Both labels must be present. Got counts: {vc.to_dict()}")
    if (vc[[case_label, control_label]] < 2).any():
        raise ValueError(f"Each group should have at least 2 samples. Got counts: {vc.to_dict()}")

def run_single_deseq(counts_df: pd.DataFrame,
                     meta_df: pd.DataFrame,
                     cfg: AnalysisConfig) -> Dict[str, object]:
    validate_subset_for_de(meta_df, cfg.CONDITION_COL, cfg.CASE_LABEL, cfg.CONTROL_LABEL)

    counts_use = filter_genes_by_min_count(counts_df, cfg.MIN_COUNT_GENE).copy()
    counts_use = counts_use.astype(int)

    meta_use = meta_df.copy()
    meta_use[cfg.CONDITION_COL] = pd.Categorical(
        meta_use[cfg.CONDITION_COL].astype(str),
        categories=[cfg.CONTROL_LABEL, cfg.CASE_LABEL],
        ordered=True,
    )

    inference = DefaultInference(n_cpus=cfg.N_CPUS)

    dds = DeseqDataSet(
        counts=counts_use,
        metadata=meta_use,
        design_factors=cfg.DESIGN,
        refit_cooks=cfg.REFIT_COOKS,
        inference=inference,
        min_replicates=cfg.MIN_REPLICATES,
        quiet=cfg.QUIET,
    )
    dds.deseq2()

    stat_res = DeseqStats(
        dds,
        contrast=[cfg.CONDITION_COL, cfg.CASE_LABEL, cfg.CONTROL_LABEL],
        alpha=cfg.ALPHA,
        cooks_filter=cfg.COOKS_FILTER,
        independent_filter=cfg.INDEPENDENT_FILTER,
        inference=inference,
        quiet=cfg.QUIET,
        n_cpus=cfg.N_CPUS,
    )
    stat_res.summary()

    res = stat_res.results_df.copy()
    res.index.name = "gene"
    res = res.reset_index()

    metric = cfg.SIGNIFICANCE_METRIC
    if metric not in {"pvalue", "padj"}:
        raise ValueError("SIGNIFICANCE_METRIC must be 'pvalue' or 'padj'")

    res["is_significant"] = res[metric].notna() & (res[metric] < cfg.ALPHA)

    return {
        "dds": dds,
        "stat_res": stat_res,
        "results_df": res,
        "n_samples": meta_use.shape[0],
        "n_genes_input": counts_df.shape[1],
        "n_genes_tested": counts_use.shape[1],
        "group_counts": meta_use[cfg.CONDITION_COL].astype(str).value_counts().to_dict(),
    }



## 6. 반복 실행 및 결과 집계


In [ ]:

def run_repeated_de_analysis(counts_df: pd.DataFrame,
                             meta_df: pd.DataFrame,
                             cfg: AnalysisConfig):
    iteration_summaries = []
    all_results = []
    errors = []

    for subset in generate_subsets(counts_df, meta_df, cfg):
        iteration_id = subset["iteration_id"]
        sub_counts = subset["counts"]
        sub_meta = subset["metadata"]

        try:
            out = run_single_deseq(sub_counts, sub_meta, cfg)
            res = out["results_df"].copy()
            res["iteration_id"] = iteration_id
            res["subset_type"] = subset["subset_type"]
            res["n_samples"] = out["n_samples"]
            res["n_genes_input"] = out["n_genes_input"]
            res["n_genes_tested"] = out["n_genes_tested"]

            group_counts = out["group_counts"]
            res["n_case"] = group_counts.get(cfg.CASE_LABEL, 0)
            res["n_control"] = group_counts.get(cfg.CONTROL_LABEL, 0)

            all_results.append(res)

            iteration_summaries.append({
                "iteration_id": iteration_id,
                "subset_type": subset["subset_type"],
                "status": "ok",
                "n_samples": out["n_samples"],
                "n_genes_input": out["n_genes_input"],
                "n_genes_tested": out["n_genes_tested"],
                "n_case": group_counts.get(cfg.CASE_LABEL, 0),
                "n_control": group_counts.get(cfg.CONTROL_LABEL, 0),
                "n_significant": int(res["is_significant"].sum()),
                "median_abs_log2fc": float(res["log2FoldChange"].abs().median(skipna=True)),
            })

        except Exception as e:
            errors.append({"iteration_id": iteration_id, "error": str(e)})
            iteration_summaries.append({
                "iteration_id": iteration_id,
                "subset_type": subset["subset_type"],
                "status": "error",
                "error": str(e),
            })

    iteration_summary_df = pd.DataFrame(iteration_summaries)
    all_results_df = pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()
    errors_df = pd.DataFrame(errors)

    return iteration_summary_df, all_results_df, errors_df

def summarize_gene_stability(all_results_df: pd.DataFrame,
                             significance_metric: str = "padj") -> pd.DataFrame:
    if all_results_df.empty:
        return pd.DataFrame()

    grouped = all_results_df.groupby("gene", dropna=False)
    summary = grouped.agg(
        runs_present=("iteration_id", "nunique"),
        sig_frequency=("is_significant", "mean"),
        mean_log2fc=("log2FoldChange", "mean"),
        std_log2fc=("log2FoldChange", "std"),
        median_log2fc=("log2FoldChange", "median"),
        mean_pvalue=("pvalue", "mean"),
        median_pvalue=("pvalue", "median"),
        mean_padj=("padj", "mean"),
        median_padj=("padj", "median"),
        mean_baseMean=("baseMean", "mean"),
        median_baseMean=("baseMean", "median"),
    ).reset_index()

    summary["abs_mean_log2fc"] = summary["mean_log2fc"].abs()
    summary = summary.sort_values(
        ["sig_frequency", "abs_mean_log2fc", "median_baseMean"],
        ascending=[False, False, False],
    ).reset_index(drop=True)
    return summary

def compute_jaccard_matrix(all_results_df: pd.DataFrame) -> pd.DataFrame:
    if all_results_df.empty:
        return pd.DataFrame()

    sig_sets = {
        it: set(df.loc[df["is_significant"], "gene"].astype(str))
        for it, df in all_results_df.groupby("iteration_id")
    }
    iterations = list(sig_sets.keys())
    mat = pd.DataFrame(index=iterations, columns=iterations, dtype=float)

    for i in iterations:
        for j in iterations:
            a, b = sig_sets[i], sig_sets[j]
            denom = len(a | b)
            mat.loc[i, j] = np.nan if denom == 0 else len(a & b) / denom

    return mat



## 7. 시각화 함수

- 반복별 유의 유전자 수 분포
- 반복별 `|log2FC|` 중앙값
- 가장 안정적으로 반복 검출되는 유전자 Top N
- 반복 간 Jaccard similarity heatmap


In [ ]:

def plot_iteration_summary(iteration_summary_df: pd.DataFrame):
    ok_df = iteration_summary_df[iteration_summary_df["status"] == "ok"].copy()
    if ok_df.empty:
        print("No successful iterations to plot.")
        return

    plt.figure(figsize=(10, 5))
    plt.bar(ok_df["iteration_id"], ok_df["n_significant"])
    plt.xticks(rotation=90)
    plt.ylabel("Number of significant genes")
    plt.title("Significant gene counts per iteration")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 5))
    plt.bar(ok_df["iteration_id"], ok_df["median_abs_log2fc"])
    plt.xticks(rotation=90)
    plt.ylabel("Median |log2FC|")
    plt.title("Median absolute log2 fold change per iteration")
    plt.tight_layout()
    plt.show()

def plot_top_stable_genes(gene_stability_df: pd.DataFrame, top_n: int = 20):
    if gene_stability_df.empty:
        print("Empty gene stability table.")
        return

    top = gene_stability_df.head(top_n).iloc[::-1]

    plt.figure(figsize=(8, max(6, top_n * 0.35)))
    plt.barh(top["gene"], top["sig_frequency"])
    plt.xlabel("Significant frequency across iterations")
    plt.ylabel("Gene")
    plt.title(f"Top {top_n} stable genes")
    plt.tight_layout()
    plt.show()

def plot_jaccard_heatmap(jaccard_df: pd.DataFrame):
    if jaccard_df.empty:
        print("Empty Jaccard matrix.")
        return

    plt.figure(figsize=(8, 6))
    plt.imshow(jaccard_df.values, aspect="auto")
    plt.colorbar(label="Jaccard similarity")
    plt.xticks(range(len(jaccard_df.columns)), jaccard_df.columns, rotation=90)
    plt.yticks(range(len(jaccard_df.index)), jaccard_df.index)
    plt.title("Jaccard similarity of significant gene sets")
    plt.tight_layout()
    plt.show()



## 8. 실제 실행 셀

아래 셀에서 경로와 파라미터를 수정한 뒤 실행하세요.


In [ ]:

cfg = AnalysisConfig(
    # COUNTS_DIR="./Dataset/RNA/IR3/counts/",
    # METADATA_PATH="./Dataset/RNA/IR3/metaDataIR3.csv",
    # RESULTS_DIR="./de_stability_results",

    # DIAGNOSES=("PD", "Control"),
    # VISITS=("BL",),
    # QC_PASS_ONLY=True,

    # ANALYSIS_MODE="kfold",
    # N_ITERATIONS_OR_FOLDS=5,
    # RANDOM_STATE=42,

    # CONDITION_COL="condition",
    # CASE_LABEL="PD",
    # CONTROL_LABEL="Control",
    # DESIGN="~condition",

    # MIN_COUNT_GENE=10,
    # SIGNIFICANCE_METRIC="padj",
    # ALPHA=0.05,

    # N_CPUS=None,
    # REFIT_COOKS=True,
    # COOKS_FILTER=True,
    # INDEPENDENT_FILTER=True,
    # MIN_REPLICATES=3,
    # KFOLD_SUBSET="train",
    # QUIET=True,
)

print(json.dumps(asdict(cfg), indent=2, ensure_ascii=False))


In [ ]:

counts_all, meta_all, matched_table = load_and_prepare_data(cfg)

print("counts_all shape:", counts_all.shape)
print("meta_all shape:", meta_all.shape)
print("\nGroup counts:")
print(meta_all[cfg.CONDITION_COL].value_counts(dropna=False))

display(matched_table.head())
display(meta_all[[cfg.CONDITION_COL, "CLINICAL_EVENT", "QCflagIR3", "PATNO", "HudAlphaID"]].head())


In [ ]:

iteration_summary_df, all_results_df, errors_df = run_repeated_de_analysis(counts_all, meta_all, cfg)

print("iteration_summary_df shape:", iteration_summary_df.shape)
print("all_results_df shape:", all_results_df.shape)
print("errors_df shape:", errors_df.shape)

display(iteration_summary_df)
if not errors_df.empty:
    display(errors_df)


In [ ]:

gene_stability_df = summarize_gene_stability(all_results_df, significance_metric=cfg.SIGNIFICANCE_METRIC)
jaccard_df = compute_jaccard_matrix(all_results_df)

print("gene_stability_df shape:", gene_stability_df.shape)
display(gene_stability_df.head(30))
display(jaccard_df)


In [ ]:

results_dir = Path(cfg.RESULTS_DIR)
results_dir.mkdir(parents=True, exist_ok=True)

matched_table.to_csv(results_dir / "matched_samples.csv", index=False)
iteration_summary_df.to_csv(results_dir / "iteration_summary.csv", index=False)
all_results_df.to_csv(results_dir / "all_gene_results.csv.gz", index=False, compression="gzip")
gene_stability_df.to_csv(results_dir / "gene_stability_summary.csv", index=False)
jaccard_df.to_csv(results_dir / "jaccard_similarity.csv")

with open(results_dir / "analysis_config.json", "w", encoding="utf-8") as f:
    json.dump(asdict(cfg), f, indent=2, ensure_ascii=False)

print(f"Saved outputs to: {results_dir.resolve()}")


In [ ]:

plot_iteration_summary(iteration_summary_df)
plot_top_stable_genes(gene_stability_df, top_n=20)
plot_jaccard_heatmap(jaccard_df)



## 9. 해석 팁

주요 결과 파일:
- `iteration_summary.csv`  
  반복별 유의 유전자 수, 테스트된 유전자 수, 샘플 수 등 요약
- `all_gene_results.csv.gz`  
  각 반복/폴드에서 각 유전자별 DE 결과 전체
- `gene_stability_summary.csv`  
  유전자별 반복 검출 빈도, 평균/표준편차 log2FC
- `jaccard_similarity.csv`  
  반복 간 유의 유전자 집합의 Jaccard 유사도

권장 해석 포인트:
1. **반복별 유의 유전자 수 분포**가 넓으면 결과가 분할에 민감할 수 있습니다.
2. **sig_frequency**가 높은 유전자는 반복적으로 검출되는 안정적인 후보입니다.
3. **std_log2fc**가 큰 유전자는 효과크기 추정이 불안정할 수 있습니다.
4. **Jaccard similarity**가 높을수록 반복 간 유의 유전자 집합이 일관적입니다.

## 10. 확장 아이디어

- `DESIGN="~ GENDER + condition"` 같은 다변량 설계
- visit를 포함한 설계 (`~ CLINICAL_EVENT + condition`)
- 환자 단위(grouped) resampling
- 유전자 안정성의 bootstrap confidence interval
- volcano plot / MA plot / PCA
- fold별 공통 유전자만 추출하는 consensus gene list
